# Export LidarScout Model to ExecuTorch
This notebook captures the `IpesCnn` model via `torch.export` (FX Tracing) and lowers it to an `.pte` file for C++ inference using ExecuTorch.

In [1]:
import os
import sys
import torch
import pytorch_lightning as pl

# ExecuTorch specific imports
from torch.export import export
from executorch.exir import to_edge

# Import your specific model class
from source.modules.ipes_cnn import IpesCnn

print(f"Python path: {sys.executable}")

W0622 20:04:16.267000 21608 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Python path: c:\repos\lidarscout_training\.venv\Scripts\python.exe


In [2]:
CHECKPOINT_PATH = r"C:\repos\lidarscout_training\models\ipes_cnn_v2_extra_data_voidloss_arch2\alpha\checkpoints\last.ckpt"
OUTPUT_EXECUTORCH_PATH = r"exported_models\ipes_cnn_v2_extra_data_voidloss_arch2\ipes_cnn_rgb.pte"
YAML_CONFIG_PATH = r"C:\repos\lidarscout_training\models\ipes_cnn_v2_extra_data_voidloss_arch2\alpha\config.yaml"

In [3]:
print(f"Loading config from {YAML_CONFIG_PATH}...")

with open(YAML_CONFIG_PATH, 'r') as file:
    import yaml
    config = yaml.safe_load(file)
    
model_kwargs = config['model']['init_args']
data_kwargs = config['data']['init_args']

model_kwargs['pts_to_img_methods'] = data_kwargs['pts_to_img_methods']
model_kwargs['hm_size'] = data_kwargs['hm_size']
model_kwargs['in_file'] = data_kwargs['in_file']

if model_kwargs.get('workers') is None:
    model_kwargs['workers'] = data_kwargs.get('workers', 0)

Loading config from C:\repos\lidarscout_training\models\ipes_cnn_v2_extra_data_voidloss_arch2\alpha\config.yaml...


In [4]:
print(f"Loading model from {CHECKPOINT_PATH}...")

network = IpesCnn.load_from_checkpoint(
    checkpoint_path=CHECKPOINT_PATH,
    **model_kwargs
)

network.eval()
print("Model loaded and set to eval mode successfully!")

Loading model from C:\repos\lidarscout_training\models\ipes_cnn_v2_extra_data_voidloss_arch2\alpha\checkpoints\last.ckpt...
Model loaded and set to eval mode successfully!


In [5]:
print("Applying cuDNN deterministic settings...")
torch.backends.cudnn.enabled = True
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

Applying cuDNN deterministic settings...


In [6]:
def make_example_data(net, batch_size=1):
    res = net.hm_interp_size
    example_inputs = dict()

    for m in net.input_methods:
        example_inputs[f'patch_hm_{m}'] = torch.rand(batch_size, 1, res, res)
        example_inputs[f'patch_rgb_{m}'] = torch.rand(batch_size, 3, res, res)
        
    example_inputs['patch_hm_mask'] = torch.rand(batch_size, 1, res, res)
    
    return example_inputs

dummy_inputs = make_example_data(network, batch_size=1)

# ExecuTorch/FX Tracing requires inputs packaged as a tuple for args, 
# or dict for kwargs, depending on your model's forward signature.
export_args = (dummy_inputs,)

print("Dummy inputs generated and formatted for export.")

Dummy inputs generated and formatted for export.


In [7]:
print(f"\nExporting model to {OUTPUT_EXECUTORCH_PATH}")
os.makedirs(os.path.dirname(OUTPUT_EXECUTORCH_PATH), exist_ok=True)

try:
    exported_program = export(network, args=export_args)
    print("1. ATen dialect graph captured successfully.")
except torch.export.ExportError as e:
    print("Export failed! FX tracing is strict about dynamic control flow.")
    raise e

edge_program = to_edge(exported_program)
print("2. Lowered to Edge dialect.")

executorch_program = edge_program.to_executorch()
print("3. Compiled to ExecuTorch program.")

with open(OUTPUT_EXECUTORCH_PATH, "wb") as f:
    f.write(executorch_program.buffer)

print(f"\nExport complete! Model saved to {OUTPUT_EXECUTORCH_PATH}")


Exporting model to exported_models\ipes_cnn_v2_extra_data_voidloss_arch2\ipes_cnn_rgb.pte
1. ATen dialect graph captured successfully.


C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


2. Lowered to Edge dialect.


c:\repos\lidarscout_training\.venv\Lib\site-packages\executorch\exir\tensor.py:84: FutureWarning: guard_size_oblivious will be removed. Consider using explicit unbacked handling     potentially utilizing guard_or_false, guard_or_true, or statically_known_true
  return guard_size_oblivious(self.stride < other.stride)


3. Compiled to ExecuTorch program.

Export complete! Model saved to exported_models\ipes_cnn_v2_extra_data_voidloss_arch2\ipes_cnn_rgb.pte
